# Naive Bayes
- GaussianNB used because the selected features are numerical.
- Same datasets are used as in the other models

In [ ]:
import pandas as pd

# Define label column
label_column = "Attack Type"

# Load datasets
train = pd.read_csv("train_data.csv")
validation = pd.read_csv("val_data.csv")
test = pd.read_csv("test_data.csv")

# Separate features and labels
train_x = train.drop(label_column, axis=1)
train_y = train[label_column]

validation_x = validation.drop(label_column, axis=1)
validation_y = validation[label_column]

test_x = test.drop(label_column, axis=1)
test_y = test[label_column]


Datasets sizes and classes


In [39]:
print("Train data:", train_x.shape)
print("Validation data:", validation_x.shape)
print("Test data:", test_x.shape)

print(train_y.value_counts())

print("Features:", list(train_x.columns))

Train data: (60000, 10)
Validation data: (20000, 10)
Test data: (20000, 10)
Attack Type
Normal Traffic    49849
DoS                4604
DDoS               3069
Port Scanning      2144
Brute Force         226
Web Attacks          55
Bots                 53
Name: count, dtype: int64
Features: ['Bwd Packet Length Std', 'Subflow Fwd Bytes', 'Flow Duration', 'Total Length of Fwd Packets', 'Init_Win_bytes_forward', 'Flow IAT Std', 'Active Mean', 'Bwd Packets/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Min']


Import libraries
- GaussianNB model
- StandardScaler, MinMaxScaler used for scaling
- SMOTE used because of data imbalance
- GridSearchCV used to test different parameter combinations

In [40]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report, 
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

Train, validation and test datasets are already created, so PredifendSplit is used.

In [41]:
# Combine train and validation sets for GridSearchCV
train_val_x = pd.concat([train_x, validation_x], axis=0)
train_val_y = pd.concat([train_y, validation_y], axis=0)

# Create a validation fold for PredefinedSplit
validation_fold = [-1] * len(train_x) + [0] * len(validation_x)
validation_split = PredefinedSplit(validation_fold)

Pipeline introduced
- Scaling
- Optional SMOTE
- GaussianNB -model

In [42]:
nb_pipeline = Pipeline([
    ("scaler", "passthrough"),
    ("smote", "passthrough"),
    ("model", GaussianNB())
])

Parameter grid defines different combinations
- StandardScaler, MinMaxScaler or no scaling
- With or without SMOTE
- Several smoothing options

Scoring
- Best model selected with accuracy
- Macro precision, macro recall and macro f1 calculated because of imbalanced classes

In [43]:
# Parameter grid for GridSearchCV
param_grid = {
    "scaler": [StandardScaler(), MinMaxScaler(), "passthrough"],
    "smote": [SMOTE(random_state=42, k_neighbors=3), "passthrough"],
    "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

# Define scoring metrics
scoring = {
    "accuracy": "accuracy",
    "macro_precision": make_scorer(precision_score, average="macro", zero_division=0),
    "macro_recall": make_scorer(recall_score, average="macro", zero_division=0),
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0)
}

GridSearchCV trains and evaluates all possible combinations (30 in total)
- 3 scaling options
- 2 SMOTE options
- 5 smoothing values

In [ ]:
# Perform GridSearchCV
grid_search = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="accuracy",
    cv=validation_split,
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

# Fit the model
grid_search.fit(train_val_x, train_val_y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...aussianNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__var_smoothing': [1e-09, 1e-08, ...], 'scaler': [StandardScaler(), MinMaxScaler(), ...], 'smote': [SMOTE(k_neigh...ndom_state=42), 'passthrough']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'macro_f1': make_scorer(f...ro_division=0), 'macro_precision': make_scorer(p...ro_division=0), 'macro_recall': make_scorer(r...ro_division=0)}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged

Print best validation parameters and accuracy

In [45]:
print("Best Validation Parameters:", grid_search.best_params_)
print("Best Validation Accuracy:", grid_search.best_score_)

Best Validation Parameters: {'model__var_smoothing': 1e-08, 'scaler': MinMaxScaler(), 'smote': 'passthrough'}
Best Validation Accuracy: 0.58905


GridSearchCV results table for all tested combinations
- Validation accuracy
- Validation macro precision, recall and F1
- Training accuracy

In [46]:
nb_results_table = pd.DataFrame(grid_search.cv_results_)

columns_to_display = [
    "rank_test_accuracy",
    "param_scaler",
    "param_smote",
    "param_model__var_smoothing",
    "mean_test_accuracy",
    "mean_test_macro_precision",
    "mean_test_macro_recall",
    "mean_test_macro_f1",
    "mean_train_accuracy"    
]

nb_results_table_display = nb_results_table[columns_to_display].sort_values(
    by="rank_test_accuracy"
).rename(columns={
    "rank_test_accuracy": "Rank",
    "param_scaler": "Scaler",
    "param_smote": "SMOTE",
    "param_model__var_smoothing": "Var Smoothing",
    "mean_test_accuracy": "Validation Accuracy",
    "mean_test_macro_precision": "Validation Macro Precision",
    "mean_test_macro_recall": "Validation Macro Recall",
    "mean_test_macro_f1": "Validation Macro F1",
    "mean_train_accuracy": "Train Accuracy"
})
nb_results_table_display


,Rank,Scaler,SMOTE,Var Smoothing,Validation Accuracy,Validation Macro Precision,Validation Macro Recall,Validation Macro F1,Train Accuracy
9,1,MinMaxScaler(),passthrough,1.000000e-08,0.58905,0.335236,0.748475,0.374734,0.582667
19,2,StandardScaler(),passthrough,1.000000e-06,0.58850,0.325988,0.742768,0.363378,0.582550
15,3,MinMaxScaler(),passthrough,1.000000e-07,0.58790,0.334256,0.748138,0.372797,0.581367
25,4,StandardScaler(),passthrough,1.000000e-05,0.58775,0.323825,0.741962,0.362143,0.581717
3,5,MinMaxScaler(),passthrough,1.000000e-09,0.58765,0.337376,0.746364,0.375375,0.581917
13,6,StandardScaler(),passthrough,1.000000e-07,0.58730,0.329985,0.742430,0.365966,0.581517
7,7,StandardScaler(),passthrough,1.000000e-08,0.58665,0.332849,0.742318,0.368307,0.580783
1,8,StandardScaler(),passthrough,1.000000e-09,0.58550,0.372356,0.744823,0.398968,0.579767
18,9,StandardScaler(),"SMOTE(k_neighbors=3, random_state=42)",1.000000e-06,0.58505,0.350688,0.745917,0.388213,0.578667
8,10,MinMaxScaler(),"SMOTE(k_neighbors=3, random_state=42)",1.000000e-08,0.58480,0.350832,0.745360,0.388234,0.578333


Best model with test data
- Selected with validation data
- Evaluated with separate test data

In [52]:
# Evaluate the best model on the test set
best_nb_model = grid_search.best_estimator_

# Make predictions
test_predictions = best_nb_model.predict(test_x)
train_predictions = best_nb_model.predict(train_x)
validation_predictions = best_nb_model.predict(validation_x)

# Calculate test metrics
test_accuracy = accuracy_score(test_y, test_predictions)
test_macro_precision = precision_score(test_y, test_predictions, average="macro", zero_division=0)
test_macro_recall = recall_score(test_y, test_predictions, average="macro", zero_division=0)
test_macro_f1 = f1_score(test_y, test_predictions, average="macro", zero_division=0)

# Calculate train and validation accuracy
train_accuracy = accuracy_score(train_y, train_predictions)
validation_accuracy = accuracy_score(validation_y, validation_predictions)

# Print results
print("Train Accuracy:", train_accuracy)
print("Validation Accuracy:", grid_search.best_score_)
print("Test Accuracy:", test_accuracy)
print("Test macro Precision:", test_macro_precision)
print("Test macro Recall:", test_macro_recall)
print("Test macro F1:", test_macro_f1)

# Print confusion matrix and classification report
class_labels = best_nb_model.classes_
print("Class order:", class_labels)
print("\nConfusion Matrix:\n", confusion_matrix(test_y, test_predictions))

print("\nClassification Report:\n", classification_report(test_y, test_predictions, zero_division=0))

Train Accuracy: 0.5827833333333333
Validation Accuracy: 0.58905
Test Accuracy: 0.581
Test macro Precision: 0.3421447496275506
Test macro Recall: 0.7167064933466534
Test macro F1: 0.37438663297266606
Class order: ['Bots' 'Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scanning'
 'Web Attacks']

Confusion Matrix:
 [[   1    3    0    0    0   14    0]
 [   0   62   13    0    0    0    0]
 [   0   34  959   28    2    0    0]
 [   1   91   87 1325   30    1    0]
 [ 777 3131  691 2097 8553 1190  177]
 [   0    7    0    2    1  705    0]
 [   0    2    0    0    1    0   15]]

Classification Report:
                 precision    recall  f1-score   support

          Bots       0.00      0.06      0.00        18
   Brute Force       0.02      0.83      0.04        75
          DDoS       0.55      0.94      0.69      1023
           DoS       0.38      0.86      0.53      1535
Normal Traffic       1.00      0.51      0.68     16616
 Port Scanning       0.37      0.99      0.54       715

# Naive Bayes summary
Best Naive Bayes model by accuracy
- MinMaxScaler
- No SMOTE
- Smoothing = 1e-08

Validation accuracy = 0.589
Test accuracy = 0.581

---

GaussianNB
- High recall values (e.g. DDoS, DoS, Port Scanning)
- Low precision for small classes (many false positives)
- Normal Traffic recall is only 0.51 (many normal traffic samples classified as attacks)


---

Overall, GaussianNB was fast to train but results are not very accurate. 

In [53]:
nb_summary = pd.DataFrame([{
    "Model": "Naive Bayes",
    "Best Parameters": grid_search.best_params_,
    "Train Accuracy": train_accuracy,
    "Validation Accuracy": grid_search.best_score_,
    "Test Accuracy": test_accuracy,
    "Test Macro Precision": test_macro_precision,
    "Test Macro Recall": test_macro_recall,
    "Test Macro F1": test_macro_f1
}])

nb_summary

,Model,Best Parameters,Train Accuracy,Validation Accuracy,Test Accuracy,Test Macro Precision,Test Macro Recall,Test Macro F1
0,Naive Bayes,"{'model__var_smoothing': 1e-08, 'scaler': MinM...",0.582783,0.58905,0.581,0.342145,0.716706,0.374387
